In [ ]:
%reload_ext autoreload
%autoreload 2

import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import xarray as xr
from bonner.computation.metrics import pearson_r
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

from lib.datasets import compute_shared_stimuli, filter_by_stimulus, things
from lib.datasets.things import load_embeddings
from lib.spectra import (
    CrossDecomposition,
    bin_data,
    compute_spectra_with_n_fold_cross_validation,
    extract_geometrically_spaced_bins,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC, mathtext_exponent_label

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

In [ ]:
spectra = []
rois = list(things.ROI_MAPPINGS.keys())
rois = ("V1", "V2", "V3", "hV4", "LOC")

datasets = {}

for roi in rois:
    datasets[roi] = {}

    for subject in range(things.N_SUBJECTS):
        dataset = things.load_dataset(subject=subject, roi=roi, z_score=False)
        datasets[roi][subject] = dataset.isel(
            presentation=dataset["index"] < 12,
        )

    stimuli = compute_shared_stimuli(datasets[roi].values())

    for subject in datasets[roi]:
        datasets[roi][subject] = filter_by_stimulus(
            datasets[roi][subject],
            stimuli=stimuli,
        )

In [ ]:
bin_edges, bin_centers = extract_geometrically_spaced_bins(
    start=1,
    stop=8_640,
    density=3,
)

fig, axes = plt.subplots(
    figsize=(6, 2.5),
    ncols=len(rois),
    sharex="row",
    sharey="row",
)

for i_roi, roi in enumerate(rois):
    spectra = []
    for (subject_1, dataset_1), (subject_2, dataset_2) in itertools.combinations(
        datasets[roi].items(),
        r=2,
    ):
        spectra_ = compute_spectra_with_n_fold_cross_validation(
            x_train=dataset_1,
            y_train=dataset_2,
            x_test=dataset_1,
            y_test=dataset_2,
            n_folds=8,
            metric="covariance",
            n_permutations=5_000,
        ).expand_dims(roi=[roi], comparison=[f"S{subject_1 + 1} & S{subject_2 + 1}"])

        spectra_ = bin_data(
            spectra_,
            bin_edges={"component": bin_edges},
            bin_centers={"component": bin_centers},
            dim="rank",
        )
        spectra.append(spectra_)

    ax = axes[i_roi]
    plot_spectra(
        spectra=xr.merge(spectra),
        ax=ax,
        hue="comparison",
        hue_order=list(reversed(["S1 & S2", "S1 & S3", "S2 & S3"])),
        hide_insignificant=True,
        marker="o",
        null_quantile=0.999,
        palette="flare_r",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(left=1, right=1.5e3)
    ax.set_ylim(bottom=1e-9, top=1e-4)
    ax.set_title(roi if roi != "hV4" else "V4")
    ax.set_xticks([1, 1e1, 1e2, 1e3])

    # xtick_exponents = list(range(5))
    # ax.set_xticks(
    #     [10**exponent for exponent in xtick_exponents],
    #     labels=[
    #         mathtext_exponent_label(exponent) if exponent in {0, 2, 4} else ""
    #         for exponent in xtick_exponents
    #     ],
    # )

    ytick_exponents = list(range(-9, -3))
    ax.set_yticks(
        [10**exponent for exponent in ytick_exponents],
        labels=[
            mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
            for exponent in ytick_exponents
        ],
    )

axes[-1].legend(reverse=True, bbox_to_anchor=(0.75, 0.85), title="comparison")

fig.supxlabel("rank", x=0.5, y=0.05)
axes[0].set_ylabel("cross-covariance")
fig.suptitle("between-subject spectra, THINGS dataset (8,640 object images)")

save_figure(fig, filepath=FIGURES_HOME / "things.pdf")

In [ ]:
cross_decomposition = CrossDecomposition(randomized=True)
cross_decomposition.fit(datasets["LOC"][0], datasets["LOC"][1])
transformed = cross_decomposition.transform(
    datasets["LOC"][0],
    direction="left",
).assign_coords(**{
    metadata: ("presentation", datasets["LOC"][0][metadata].to_numpy())
    for metadata in ("stimulus", "object")
})
transformed = transformed.groupby("object").mean()

In [ ]:
behavioral_embedding = load_embeddings().sel(object=transformed["object"])
correlations = pearson_r(
    torch.from_numpy(behavioral_embedding.to_numpy()).to(torch.float32),
    torch.from_numpy(transformed.to_numpy()).to(torch.float32),
    return_diagonal=False,
)

fig, axes = plt.subplots(figsize=(6.5, 7.5), nrows=2, height_ratios=[5, 1])

ax = axes[0]
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.25)

sns.heatmap(
    ax=ax,
    data=correlations[:, :100].abs(),
    vmin=0,
    vmax=0.5,
    cmap="afmhot",
    # square=True,
    yticklabels=behavioral_embedding["behavior"].to_numpy(),
    xticklabels=False,
    cbar_ax=cax,
)
ax.tick_params(axis="y", length=0, width=0, labelsize="xx-small")
ax.tick_params(axis="x", length=0, width=0, labelsize="x-small")
# ax.set_xticks([1, 10, 100])
dimensions = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
ax.set_xticks([d - 0.5 for d in dimensions], dimensions)
cax.set_ylabel("Pearson correlation (absolute)", labelpad=20, rotation=-90)
ax.set_title(
    "correlations between behavioral dimensions and neural dimensions (THINGS dataset)",
    pad=20,
    fontsize="medium",
)

ax = axes[1]
sns.lineplot(
    ax=ax,
    data=(
        pd.DataFrame(
            correlations.abs(),
            index=behavioral_embedding["behavior"].to_numpy(),
            columns=np.arange(correlations.shape[-1]),
        )
        .melt(ignore_index=False, var_name="neural_dimension", value_name="correlation")
        .reset_index(names="behavioral_dimension")
        .groupby("neural_dimension")
        .max()
        .reset_index()
    ),
    x="neural_dimension",
    y="correlation",
    c="k",
    lw=0.5,
)
ax.axhspan(0, 0.5, xmin=0, xmax=0.2, facecolor="gainsboro")
ax.set_xlim(left=0, right=500)
ax.set_ylim(bottom=0, top=0.5)
ax.set_ylabel(
    "maximum Pearson correlation\nacross behavioral dimensions\n(absolute)",
    rotation=0,
    va="center",
    ha="center",
    labelpad=75,
)
ax.set_xlabel(
    "neural dimensions (lateral occipital complex)\nleft singular vectors, cross-decomposition between subjects 1 and 2",
)
# fig.set_facecolor("w")
save_figure(fig, filepath=FIGURES_HOME / "things-dimensions.pdf")